<a href="https://colab.research.google.com/github/jawad-glitch/uncertainty-aware-drone-nav/blob/main/notebooks/kf_imu_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kalman filters with IMU: from constant-velocity to IMU + accelerometer bias

Does not run the simulator: it loads the saved ground-truth flight (`results/pid_flight.npz`),
adds seeded GPS and IMU noise, and compares three filters on the same data.

| Filter | Predict step | State |
|---|---|---|
| const-vel | assumes velocity stays constant | `[x, y, vx, vy]` |
| IMU | driven by the accelerometer | `[x, y, vx, vy]` |
| IMU + bias | driven by accelerometer minus estimated bias | `[x, y, vx, vy, bx, by]` |

All three skip the GPS update during dropouts. The IMU rotation uses the **true** attitude, so this is
still a stepping stone to the EKF, which estimates attitude from the gyro.

**Colab:** run the first code cell. **Locally:** skip it (repo and venv already set up) and start at the second.

In [ ]:
!git clone https://github.com/jawad-glitch/uncertainty-aware-drone-nav.git
%cd uncertainty-aware-drone-nav
!pip install -r requirements.txt

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import sensors, filters

data = dict(np.load(ROOT / "results" / "pid_flight.npz"))
meas = sensors.make_sensors(data, seed=0)
dt, valid = 1/48, meas["gps_valid"]

# accelerometer readings back to world-frame acceleration (uses true attitude)
a_world = filters.a_world_from_imu(meas["accel"], data["rpy"])

est_cv,  sig_cv  = filters.run_cv_kf(meas["gps"], valid, dt, q=1e-4)
est_imu, sig_imu = filters.run_imu_kf(a_world, meas["gps"], valid, dt)
est_b,   sig_b   = filters.run_imu_bias_kf(a_world, meas["gps"], valid, dt)

In [ ]:
# sanity check: a_world should match the true acceleration up to noise + bias (~0.1 m/s^2)
a_true = np.gradient(data["vel"], data["time"], axis=0)
print("accel error RMS (x, y):", np.sqrt(np.mean((a_world[:, :2] - a_true[:, :2])**2, axis=0)))
print("mean a_world z (should be ~0):", a_world[:, 2].mean())

In [ ]:
t = data["time"]
true_xy = data["pos"][:, :2]

def pos_err(est):
    return np.linalg.norm(est[:, :2] - true_xy, axis=1)

err_cv, err_imu, err_b = pos_err(est_cv), pos_err(est_imu), pos_err(est_b)
err_x_b = est_b[:, 0] - true_xy[:, 0]

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

# 1) error of all three filters
ax[0].plot(t, err_cv, alpha=0.5, label="const-vel")
ax[0].plot(t, err_imu, alpha=0.5, label="IMU, no bias")
ax[0].plot(t, err_b, label="IMU + bias")
ax[0].fill_between(t, 0, 1, where=~valid, transform=ax[0].get_xaxis_transform(),
                   color="red", alpha=0.2, label="GPS dropout")
ax[0].set_xlabel("time (s)")
ax[0].set_ylabel("position error (m)")
ax[0].legend()
ax[0].set_title("error vs truth")

# 2) can the filter find the accelerometer bias?
ax[1].plot(t, meas["accel_bias"][:, 0], label="true bias x")
ax[1].plot(t, est_b[:, 4], label="estimated bias x")
ax[1].fill_between(t, 0, 1, where=~valid, transform=ax[1].get_xaxis_transform(),
                   color="red", alpha=0.2)
ax[1].set_xlabel("time (s)")
ax[1].set_ylabel("m/s²")
ax[1].legend()
ax[1].set_title("accelerometer bias")

# 3) does the filter know how wrong it is?
ax[2].plot(t, err_x_b, label="x error")
ax[2].fill_between(t, -3 * sig_b[:, 0], 3 * sig_b[:, 0], alpha=0.25, label="±3σ")
ax[2].fill_between(t, 0, 1, where=~valid, transform=ax[2].get_xaxis_transform(),
                   color="red", alpha=0.2)
ax[2].set_xlabel("time (s)")
ax[2].set_ylabel("x error (m)")
ax[2].legend()
ax[2].set_title("IMU + bias: error vs uncertainty")

plt.tight_layout()
plt.show()

rmse = filters.rmse
print("                 const-vel   IMU    IMU+bias")
print(f"all steps:       {rmse(err_cv):.4f}     {rmse(err_imu):.4f}   {rmse(err_b):.4f}")
print(f"GPS steps:       {rmse(err_cv, valid):.4f}     {rmse(err_imu, valid):.4f}   {rmse(err_b, valid):.4f}")
print(f"inside dropout:  {rmse(err_cv, ~valid):.4f}     {rmse(err_imu, ~valid):.4f}   {rmse(err_b, ~valid):.4f}")

In [ ]:
# Diagnostic: how good could the IMU filter be with a perfect IMU (no noise, no bias)?
# Same seed, so the GPS is identical; only the IMU changes.
meas_clean = sensors.make_sensors(data, seed=0, accel_noise=0,
                                  accel_bias_walk=0, accel_bias_init=0)
a_clean = filters.a_world_from_imu(meas_clean["accel"], data["rpy"])
est_clean, _ = filters.run_imu_kf(a_clean, meas["gps"], valid, dt, q=1e-8)
err_clean = pos_err(est_clean)
print("perfect IMU, inside dropout:", rmse(err_clean, ~valid))
print("perfect IMU, GPS steps:     ", rmse(err_clean, valid))